# 04 · Correlations (Apriori) + Next-Item Prediction (KNN)
**Owner: P3 (AI)** · Smart Shelf Market Basket Analysis

1. Simulate customer baskets over the shelf's products with **planted "impossible" correlations**
2. **Apriori** - mine association rules; how many planted rules are recovered?
3. **Experiment 4** - pass baskets through the detector's error profile (missed/spurious items) and measure
   how rule recovery degrades as detection recall drops → *how good must the vision system be?*
4. **Experiment 5** - **KNN** next-item recommendation (Hit@3), Hamming vs Jaccard, vs popularity baseline
5. Apply to the real shelf baskets from notebook 03

Runs on any laptop, CPU only, ~1-2 minutes.

In [ ]:
# ============================================================
# SETUP - identical in every notebook. Works on Colab, Mac, Windows.
# ============================================================
import os, sys, subprocess
from pathlib import Path

# Colab only: where to get the code from.
#   - GitHub (recommended): set with  python tools/set_repo_url.py <url>
#   - "" : the product-identification-correlation folder is in Google Drive at MyDrive/product-identification-correlation
REPO_URL = "https://github.com/behlah-rgb/product-identification-correlation.git"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    if REPO_URL:
        repo = Path("/content/product-identification-correlation")
        url = REPO_URL
        try:  # private repo: add a GitHub token as Colab secret GITHUB_TOKEN
            from google.colab import userdata
            url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
        except Exception:
            pass
        if repo.exists():
            subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
        else:
            subprocess.run(["git", "clone", "-q", url, str(repo)], check=True)
    else:
        repo = Path("/content/drive/MyDrive/product-identification-correlation")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)
else:
    repo = Path.cwd()
    while not (repo / "src").exists() and repo != repo.parent:
        repo = repo.parent

sys.path.insert(0, str(repo))
os.chdir(repo)

from src import env
env.summary()

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import basket_sim as bs, mining
from src.demo_data import DEMO_CLASSES
from src.env import DATASETS_DIR, RESULTS_DIR
from src.detection import read_names

DATASET = "indian_grocery"
ds = DATASETS_DIR / DATASET
PRODUCTS = read_names(ds) if (ds / "data.yaml").exists() else DEMO_CLASSES
print("Products:" if PRODUCTS is not DEMO_CLASSES else "Products (DEMO names - download the dataset in 02):", PRODUCTS)

## 1. Simulate baskets with planted rules
Edit `PLANTED` to choose which correlations to hide. Keep a mix of *expected* and *"impossible"*
(cross-category) pairs - that's the beers-and-diapers angle of the paper.

In [ ]:
P = PRODUCTS
PLANTED = bs.default_planted_rules(P)
# Example of writing your own (use exact class names):
# PLANTED = [bs.PlantedRule(("nescafe",), "nivea_lotion", 0.5), ...]
for r in PLANTED:
    print("planted:", r)

N_BASKETS = 2000
baskets, base = bs.simulate_baskets(P, n_baskets=N_BASKETS, planted=PLANTED, seed=42)
matrix = mining.to_matrix(baskets, items=P)

fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
pd.Series([len(b) for b in baskets]).value_counts().sort_index().plot.bar(ax=ax[0], rot=0)
ax[0].set_title("Basket size"); ax[0].set_xlabel("items")
matrix.mean().sort_values().plot.barh(ax=ax[1]); ax[1].set_title("Item support")
plt.tight_layout(); plt.show()
matrix.astype(int).head()

## 2. Apriori on clean baskets

In [ ]:
MIN_SUPPORT, MIN_LIFT, MIN_CONF = 0.005, 1.2, 0.1

itemsets, rules = mining.mine_rules(matrix, MIN_SUPPORT, MIN_LIFT, MIN_CONF)
print(f"{len(itemsets)} frequent itemsets, {len(rules)} rules (lift >= {MIN_LIFT})")
clean_rec = mining.rule_recovery(rules, PLANTED)
print(f"Planted rules recovered: {clean_rec['recall']:.0%} | precision@{clean_rec['k']}: {clean_rec['precision_at_k']:.0%}")
mining.readable(rules, 10)

## 3. Experiment 4 - effect of detection errors on discovered correlations
Error profile comes from notebook 02's per-class recall/precision (if a YOLO result exists),
otherwise a default of recall 0.9, 5% spurious items.

In [ ]:
yolo_json = RESULTS_DIR / "yolo_public_only.json"
if yolo_json.exists():
    prof = bs.noise_profile_from_yolo(yolo_json)
    print("Using detector error profile from", yolo_json.name, "| fp_rate =", round(prof["fp_rate"], 3))
else:
    prof = {"recall": 0.9, "fp_rate": 0.05}
    print("No YOLO results yet - default profile", prof)

noisy = bs.apply_detection_noise(baskets, P, recall=prof["recall"], fp_rate=prof["fp_rate"], seed=1)
noisy_matrix = mining.to_matrix(noisy, items=P)
_, noisy_rules = mining.mine_rules(noisy_matrix, MIN_SUPPORT, MIN_LIFT, MIN_CONF)
noisy_rec = mining.rule_recovery(noisy_rules, PLANTED)

pd.DataFrame([
    {"baskets": "clean (ground truth)", "rules": len(rules), "planted recovered": clean_rec["recall"], f"precision@{clean_rec['k']}": clean_rec["precision_at_k"]},
    {"baskets": "through detector", "rules": len(noisy_rules), "planted recovered": noisy_rec["recall"], f"precision@{clean_rec['k']}": noisy_rec["precision_at_k"]},
]).round(3)

In [ ]:
# Sweep detector recall (and spurious-item rate) - averaged over 3 seeds
def planted_lift(m, rule):
    s = list(rule.itemset())
    joint = m[s].all(axis=1).mean()
    a = m[list(rule.antecedent)].all(axis=1).mean(); c = m[rule.consequent].mean()
    return joint / (a * c) if a * c else np.nan

sweep = []
for rec in [1.0, 0.95, 0.9, 0.85, 0.8, 0.7, 0.6, 0.5]:
    for fp in [0.0, 0.1, 0.2]:
        res = []
        for seed in range(3):
            nb = bs.apply_detection_noise(baskets, P, recall=rec, fp_rate=fp, seed=seed)
            nm = mining.to_matrix(nb, items=P)
            _, nr = mining.mine_rules(nm, MIN_SUPPORT, MIN_LIFT, MIN_CONF)
            rr = mining.rule_recovery(nr, PLANTED)
            res.append((rr["recall"], rr["precision_at_k"], np.nanmean([planted_lift(nm, r) for r in PLANTED])))
        r_, p_, l_ = np.mean(res, axis=0)
        sweep.append({"detector_recall": rec, "fp_rate": fp, "rules_recovered": r_, "precision_at_k": p_, "mean_planted_lift": l_})
sweep = pd.DataFrame(sweep)
sweep.to_csv(RESULTS_DIR / "exp4_noise_sweep.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for fp, g in sweep.groupby("fp_rate"):
    ax[0].plot(g.detector_recall, g.rules_recovered, marker="o", label=f"spurious rate {fp}")
    ax[1].plot(g.detector_recall, g.mean_planted_lift, marker="o", label=f"spurious rate {fp}")
ax[0].set_title("Planted rules recovered by Apriori"); ax[0].set_xlabel("detector recall"); ax[0].invert_xaxis(); ax[0].legend()
ax[1].set_title("Mean lift of planted rules (signal strength)"); ax[1].set_xlabel("detector recall"); ax[1].invert_xaxis()
ax[1].axhline(1, ls="--", c="gray")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "exp4_noise_sweep.png", dpi=150); plt.show()

## 4. Experiment 5 - KNN next-item prediction
Hide one item from each test basket; recommend 3 items from the *k* most similar training baskets.
**Hit@3** = how often the hidden item is among the 3 recommendations.

In [ ]:
rng = np.random.default_rng(0)
idx = rng.permutation(len(matrix)); cut = int(0.8 * len(matrix))
train_m, test_m = matrix.iloc[idx[:cut]], matrix.iloc[idx[cut:]]

rows = [{"model": "Popularity baseline", "k": "-", "Hit@1": mining.popularity_hit_rate(train_m, test_m, top_n=1),
         "Hit@3": mining.popularity_hit_rate(train_m, test_m, top_n=3)}]
for metric in ["hamming", "jaccard"]:
    for k in [3, 5, 10, 20, 50]:
        rows.append({"model": f"KNN ({metric})", "k": k, "Hit@1": mining.knn_hit_rate(train_m, test_m, k, metric, top_n=1),
                     "Hit@3": mining.knn_hit_rate(train_m, test_m, k, metric, top_n=3)})
exp5 = pd.DataFrame(rows).round(3)
exp5.to_csv(RESULTS_DIR / "exp5_knn.csv", index=False)
exp5

In [ ]:
# Same test, but KNN trained on baskets that went through the detector (realistic deployment)
noisy_train = mining.to_matrix(bs.apply_detection_noise(train_m.apply(lambda r: list(r[r].index), axis=1).tolist(),
                                                        P, recall=prof["recall"], fp_rate=prof["fp_rate"], seed=3), items=P)
best_row = exp5[exp5.model != "Popularity baseline"].sort_values("Hit@3").iloc[-1]
metric, k = best_row.model.split("(")[1][:-1], int(best_row.k)
print(f"Best config: {metric}, k={k}")
print(f"Hit@3 clean training data : {mining.knn_hit_rate(train_m, test_m, k, metric):.3f}")
print(f"Hit@3 detector-noisy data : {mining.knn_hit_rate(noisy_train, test_m, k, metric):.3f}")

In [ ]:
# Worked example: a customer's current basket -> 3 most similar past baskets -> suggestions
knn = mining.fit_knn(train_m, n_neighbors=3, metric="hamming")
current = [P[0], P[2]]
recs, neigh, dist = mining.recommend(knn, train_m, current, top_n=3)
print("Current basket :", current)
for i, (nbk, d) in enumerate(zip(neigh, dist), 1):
    print(f"  similar basket #{i} (hamming={d:.2f}): {nbk}")
print("Recommend next :", recs)

## 5. Real shelf baskets from notebook 03

In [ ]:
db = RESULTS_DIR / "detected_baskets.csv"
if not db.exists():
    print("No detected_baskets.csv yet - run notebook 03.")
else:
    shelf = pd.read_csv(db, dtype=str).fillna("")
    col = "yolo" if "yolo" in shelf and (shelf["yolo"] != "").any() else "classical"
    knn = mining.fit_knn(matrix, n_neighbors=10, metric="jaccard")
    out = []
    for _, s in shelf.iterrows():
        items = [i for i in s[col].split(";") if i in P]
        recs = mining.recommend(knn, matrix, items, 3)[0] if items else []
        out.append({"session": s.session_id, f"detected ({col})": items, "recommended next": recs})
    print("Source:", shelf.source.iloc[0])
    display(pd.DataFrame(out))

In [ ]:
summary = {"n_baskets": N_BASKETS, "min_support": MIN_SUPPORT, "min_lift": MIN_LIFT,
           "planted": [str(r) for r in PLANTED], "clean_recovery": clean_rec, "noisy_recovery": noisy_rec,
           "noise_profile_fp_rate": prof["fp_rate"]}
json.dump(summary, open(RESULTS_DIR / "exp4_summary.json", "w"), indent=2, default=str)
print("Saved results to", RESULTS_DIR)

**For the paper (P3):** Apriori settings, planted rules table, clean vs noisy recovery table, the noise-sweep
figure (key IVP→AI result), and the KNN Hit@3 table with the popularity baseline. Explain why Jaccard suits
sparse baskets better than Hamming (Hamming counts shared *absences* as similarity).